In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# 1. Cargar dataset
df = pd.read_csv("/datasets/everpeak_retail.csv")

In [ ]:
# 2. Conteo de valores faltantes
cols = ["payment_method", "city", "state", "customer_age"]
for col in cols:
    print(f"{col} missing:", df[col].isna().sum())

In [ ]:
# 3. Validación de fechas
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
invalid_year_2026_count = (df["order_date"].dt.year == 2026).sum()
missing_order_date_count = df["order_date"].isna().sum()
print("order_date año 2026:", invalid_year_2026_count)
print("order_date missing:", missing_order_date_count)

In [ ]:
# 4. Imputación de customer_age
before_age = df["customer_age"].dropna().mean()
df["customer_age_med"] = df["customer_age"].fillna(df["customer_age"].median())
df["customer_age_mean"] = df["customer_age"].fillna(df["customer_age"].mean())
print("Media original:", before_age)
print("Media imputando mediana:", df["customer_age_med"].mean())
print("Media imputando media:", df["customer_age_mean"].mean())

In [ ]:
# 5. Reconstrucción de quantity
df.loc[df['quantity'] <= 0, 'quantity'] = np.nan
df['quantity'] = df['quantity'].fillna(df['order_value'] / df['price'])

In [ ]:
# 6. Cardinalidad de columnas
print("customer_id nunique:", df["customer_id"].nunique())
print("customer_segment nunique:", df["customer_segment"].nunique())
print("city nunique:", df["city"].nunique())
print("state nunique:", df["state"].nunique())

In [ ]:

# 7. Patrones de missingness
missing_city_by_pay = df["city"].isna().groupby(df["payment_method"]).mean()
print(missing_city_by_pay)

In [ ]:
# 8. Comparar impacto entre drop e imputación en order_value
before = df["order_value"].dropna().mean()
df["order_value_imputed"] = df["order_value"].fillna(df["order_value"].median())
after = df["order_value_imputed"].mean()
print("Media dropna:", before)
print("Media imputada:", after)

In [ ]:
# 9. Nuevo análisis: detectar outliers con IQR
Q1 = df['order_value'].quantile(0.25)
Q3 = df['order_value'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5*IQR
df['outlier_flag'] = df['order_value'] > upper